# The History of Information Retrieval

This notebook walks through the evolution of information retrieval. From simple string matching to modern hybrid approaches. Each section implements a technique from scratch using numpy and scipy so you can see exactly how these methods work under the hood.

## The Evolution at a Glance

| Era | Method | Core Idea |
|-----|--------|----------|
| 1960s | Levenshtein Distance | How many edits to turn one string into another? |
| 1970s | Jaccard Similarity | How much do two sets of words overlap? |
| 1970-90s | TF-IDF & Word Frequency | Rare words matter more than common ones |
| 2013+ | Word Vectors (Word2Vec) | Words live in a continuous space — similar words are nearby |
| 2018+ | Neural Ranking | Deep neural networks score query-document relevance directly |
| 2020s | Hybrid Retrieval | Combine keyword + semantic search for the best of both worlds |

Let's build each one from scratch.

In [1]:
%pip install -q --upgrade numpy scipy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.21.3 requires pyarrow<20,>=4.0.0, but you have pyarrow 20.0.0 which is incompatible.
lightning-whisper-mlx 0.0.10 requires tiktoken==0.3.3, but you have tiktoken 0.12.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from scipy.spatial.distance import cosine
import re
from collections import Counter

## Our Toy Corpus

We'll use the same small corpus throughout so you can compare how each method retrieves documents for the same queries.

In [3]:
# A small corpus about animals
documents = [
    "The cat sat on the mat",
    "The dog chased the cat in the garden",
    "A kitten is a young cat",
    "Dogs are loyal and friendly pets",
    "The mat was soft and warm",
    "Puppies and kittens are adorable baby animals",
    "The garden was full of flowers and birds",
    "Cats and dogs are the most popular pets worldwide",
]

def tokenize(text: str) -> list[str]:
    """Simple lowercase tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())

print(f"Corpus size: {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"  [{i}] {doc}")

Corpus size: 8 documents
  [0] The cat sat on the mat
  [1] The dog chased the cat in the garden
  [2] A kitten is a young cat
  [3] Dogs are loyal and friendly pets
  [4] The mat was soft and warm
  [5] Puppies and kittens are adorable baby animals
  [6] The garden was full of flowers and birds
  [7] Cats and dogs are the most popular pets worldwide


---

## 1. Levenshtein Distance (1960s)

The Levenshtein distance counts the minimum number of single-character edits (insertions, deletions, or substitutions) needed to change one string into another. Vladimir Levenshtein introduced this in 1965.

**Why it matters for retrieval:** It lets you find documents even when the query has typos or spelling variations. If a user searches for "cta" instead of "cat", Levenshtein distance can still match.

We build up a matrix where `dp[i][j]` = the edit distance between the first `i` characters of string A and the first `j` characters of string B.

In [4]:
def levenshtein_distance(s1: str, s2: str) -> int:
    """Compute the Levenshtein (edit) distance between two strings using dynamic programming."""
    m, n = len(s1), len(s2)
    
    # Create a matrix of size (m+1) x (n+1)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    # Base cases: transforming empty string to/from a prefix
    for i in range(m + 1):
        dp[i][0] = i  # delete all chars from s1
    for j in range(n + 1):
        dp[0][j] = j  # insert all chars into s1
    
    # Fill the matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # characters match, no edit needed
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # deletion
                    dp[i][j - 1],      # insertion
                    dp[i - 1][j - 1]   # substitution
                )
    
    return dp[m][n]

# Let's see it in action
pairs = [("cat", "cat"), ("cat", "cta"), ("cat", "cats"), ("kitten", "sitting"), ("dog", "cat")]

for a, b in pairs:
    dist = levenshtein_distance(a, b)
    print(f"  '{a}' -> '{b}' = {dist} edit(s)")

  'cat' -> 'cat' = 0 edit(s)
  'cat' -> 'cta' = 2 edit(s)
  'cat' -> 'cats' = 1 edit(s)
  'kitten' -> 'sitting' = 3 edit(s)
  'dog' -> 'cat' = 3 edit(s)


In [5]:
def levenshtein_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Search documents by finding the closest Levenshtein match for each query word.
    
    For each word in the query, we find the minimum edit distance to any word
    in each document. Lower total distance = better match.
    """
    query_words = tokenize(query)
    scores = []
    
    for idx, doc in enumerate(documents):
        doc_words = tokenize(doc)
        total_distance = 0
        
        for qw in query_words:
            # Find the closest word in the document
            min_dist = min(levenshtein_distance(qw, dw) for dw in doc_words)
            total_distance += min_dist
        
        # Normalize by number of query words (lower is better)
        avg_dist = total_distance / len(query_words)
        scores.append((idx, avg_dist, doc))
    
    # Sort by distance (ascending — lower is better)
    scores.sort(key=lambda x: x[1])
    return scores[:top_k]

# Test: exact match
print("Query: 'cat mat'")
for idx, dist, doc in levenshtein_search("cat mat", documents):
    print(f"  [{idx}] dist={dist:.2f} | {doc}")

# Test: typo in query
print("\nQuery: 'cta' (typo for 'cat')")
for idx, dist, doc in levenshtein_search("cta", documents):
    print(f"  [{idx}] dist={dist:.2f} | {doc}")

Query: 'cat mat'
  [0] dist=0.00 | The cat sat on the mat
  [1] dist=0.50 | The dog chased the cat in the garden
  [2] dist=0.50 | A kitten is a young cat

Query: 'cta' (typo for 'cat')
  [0] dist=2.00 | The cat sat on the mat
  [1] dist=2.00 | The dog chased the cat in the garden
  [2] dist=2.00 | A kitten is a young cat


**Key takeaway:** Levenshtein distance is great for fuzzy matching and handling typos, but it only compares character-level similarity. The words "dog" and "puppy" have a large edit distance even though they mean almost the same thing.

---

## 2. Jaccard Similarity (1970s)

Jaccard similarity measures the overlap between two sets. For text, we treat each document and query as a *set of words* and compute:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

It was originally proposed by Paul Jaccard in 1901 for comparing botanical samples, but became widely used in information retrieval in the 1970s.

**Why it matters:** It's simple, fast, and gives a normalized score between 0 (no overlap) and 1 (identical). It doesn't care about word frequency — just whether a word is present or absent.

In [6]:
def jaccard_similarity(set_a: set, set_b: set) -> float:
    """Compute Jaccard similarity between two sets."""
    intersection = set_a & set_b
    union = set_a | set_b
    if not union:
        return 0.0
    return len(intersection) / len(union)

def jaccard_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Rank documents by Jaccard similarity with the query."""
    query_set = set(tokenize(query))
    scores = []
    
    for idx, doc in enumerate(documents):
        doc_set = set(tokenize(doc))
        sim = jaccard_similarity(query_set, doc_set)
        scores.append((idx, sim, doc))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Demonstrate with individual set comparisons first
q = set(tokenize("cat on the mat"))
d0 = set(tokenize(documents[0]))
d1 = set(tokenize(documents[1]))

print(f"Query words:  {q}")
print(f"Doc 0 words:  {d0}")
print(f"Intersection: {q & d0}")
print(f"Union:        {q | d0}")
print(f"Jaccard:      {jaccard_similarity(q, d0):.3f}")
print()
print(f"Doc 1 words:  {d1}")
print(f"Intersection: {q & d1}")
print(f"Jaccard:      {jaccard_similarity(q, d1):.3f}")

Query words:  {'mat', 'the', 'cat', 'on'}
Doc 0 words:  {'on', 'the', 'cat', 'sat', 'mat'}
Intersection: {'the', 'cat', 'on', 'mat'}
Union:        {'on', 'the', 'cat', 'sat', 'mat'}
Jaccard:      0.800

Doc 1 words:  {'the', 'in', 'chased', 'cat', 'dog', 'garden'}
Intersection: {'cat', 'the'}
Jaccard:      0.250


In [7]:
# Full search
print("Query: 'cat on the mat'")
for idx, score, doc in jaccard_search("cat on the mat", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'loyal friendly pets'")
for idx, score, doc in jaccard_search("loyal friendly pets", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'cat on the mat'
  [0] score=0.800 | The cat sat on the mat
  [1] score=0.250 | The dog chased the cat in the garden
  [4] score=0.250 | The mat was soft and warm

Query: 'loyal friendly pets'
  [3] score=0.500 | Dogs are loyal and friendly pets
  [7] score=0.091 | Cats and dogs are the most popular pets worldwide
  [0] score=0.000 | The cat sat on the mat


**Key takeaway:** Jaccard is simple and intuitive, but it treats all words equally. The word "the" counts just as much as "kitten". It also ignores word frequency — a document mentioning "cat" 10 times scores the same as one mentioning it once.

---

## 3. TF-IDF: Word Frequency Meets Rarity (1970s-1990s)

TF-IDF (Term Frequency — Inverse Document Frequency) was one of the biggest leaps in retrieval. Instead of treating all words equally, it says: *a word that appears often in a document but rarely across the whole corpus is probably important.*

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

Where:
- **TF(t, d)** = how often term `t` appears in document `d`
- **IDF(t)** = log(total documents / documents containing `t`)

We represent each document as a vector of TF-IDF scores, then use cosine similarity to compare query and document vectors.

In [8]:
class TFIDFRetriever:
    """TF-IDF retrieval from scratch using numpy."""
    
    def __init__(self):
        self.vocab = []         # list of all unique words
        self.word_to_idx = {}   # word -> index in vocab
        self.idf = None         # IDF scores
        self.tfidf_matrix = None  # documents x vocab TF-IDF matrix
    
    def fit(self, documents: list[str]):
        """Build the TF-IDF matrix."""
        self.documents = documents
        tokenized = [tokenize(doc) for doc in documents]
        
        # Build vocabulary
        all_words = set()
        for tokens in tokenized:
            all_words.update(tokens)
        self.vocab = sorted(all_words)
        self.word_to_idx = {w: i for i, w in enumerate(self.vocab)}
        
        n_docs = len(documents)
        n_terms = len(self.vocab)
        
        # Compute Term Frequency matrix
        tf = np.zeros((n_docs, n_terms))
        for doc_idx, tokens in enumerate(tokenized):
            counts = Counter(tokens)
            for word, count in counts.items():
                tf[doc_idx, self.word_to_idx[word]] = count
        
        # Normalize TF by document length
        doc_lengths = tf.sum(axis=1, keepdims=True)
        doc_lengths[doc_lengths == 0] = 1  # avoid division by zero
        tf = tf / doc_lengths
        
        # Compute IDF
        doc_freq = (tf > 0).sum(axis=0)  # number of docs containing each term
        self.idf = np.log(n_docs / (doc_freq + 1)) + 1  # smoothed IDF
        
        # TF-IDF = TF * IDF
        self.tfidf_matrix = tf * self.idf
        
        print(f"Vocabulary size: {n_terms}")
        print(f"TF-IDF matrix shape: {self.tfidf_matrix.shape} (docs x terms)")
    
    def _query_vector(self, query: str) -> np.ndarray:
        """Convert a query string into a TF-IDF vector."""
        tokens = tokenize(query)
        vec = np.zeros(len(self.vocab))
        counts = Counter(tokens)
        for word, count in counts.items():
            if word in self.word_to_idx:
                vec[self.word_to_idx[word]] = count
        # Normalize TF
        total = vec.sum()
        if total > 0:
            vec = vec / total
        # Apply IDF
        vec = vec * self.idf
        return vec
    
    def search(self, query: str, top_k: int = 3) -> list[tuple[int, float, str]]:
        """Search using cosine similarity between query and document TF-IDF vectors."""
        q_vec = self._query_vector(query)
        
        scores = []
        for idx in range(len(self.documents)):
            d_vec = self.tfidf_matrix[idx]
            # Cosine similarity (scipy.cosine returns distance, so 1 - distance)
            if np.linalg.norm(q_vec) == 0 or np.linalg.norm(d_vec) == 0:
                sim = 0.0
            else:
                sim = 1 - cosine(q_vec, d_vec)
            scores.append((idx, sim, self.documents[idx]))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

tfidf = TFIDFRetriever()
tfidf.fit(documents)

Vocabulary size: 35
TF-IDF matrix shape: (8, 35) (docs x terms)


In [9]:
print("Query: 'cat mat'")
for idx, score, doc in tfidf.search("cat mat"):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'loyal friendly pets'")
for idx, score, doc in tfidf.search("loyal friendly pets"):
    print(f"  [{idx}] score={score:.3f} | {doc}")

# Show that TF-IDF down-weights common words
print("\nIDF scores for selected words:")
for word in ["the", "cat", "kitten", "loyal", "adorable"]:
    if word in tfidf.word_to_idx:
        print(f"  '{word}': {tfidf.idf[tfidf.word_to_idx[word]]:.3f}")

Query: 'cat mat'
  [0] score=0.523 | The cat sat on the mat
  [4] score=0.317 | The mat was soft and warm
  [1] score=0.177 | The dog chased the cat in the garden

Query: 'loyal friendly pets'
  [3] score=0.803 | Dogs are loyal and friendly pets
  [7] score=0.165 | Cats and dogs are the most popular pets worldwide
  [0] score=0.000 | The cat sat on the mat

IDF scores for selected words:
  'the': 1.288
  'cat': 1.693
  'kitten': 2.386
  'loyal': 2.386
  'adorable': 2.386


**Key takeaway:** TF-IDF is a big improvement — it automatically down-weights common words like "the" and up-weights distinctive terms like "kitten" or "loyal". But it still treats every word as completely independent. "Cat" and "kitten" have zero similarity in TF-IDF, even though they're closely related concepts.

---

## 4. Word Vectors (2013+)

Word2Vec (Mikolov et al., 2013) was a breakthrough: instead of treating words as independent symbols, it maps each word to a dense vector where *semantically similar words end up close together*.

Training real word vectors requires a large corpus and significant compute. For this toy example, we'll create hand-crafted vectors that illustrate the concept. Think of each dimension as capturing a rough semantic feature.

In [10]:
# Hand-crafted word vectors to illustrate the concept.
# Dimensions roughly correspond to: [animal, pet, young, location, soft/warm, action, plant]
word_vectors = {
    "cat":      np.array([0.9, 0.8, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "kitten":   np.array([0.9, 0.8, 0.9, 0.0, 0.3, 0.0, 0.0]),
    "dog":      np.array([0.9, 0.9, 0.0, 0.0, 0.1, 0.3, 0.0]),
    "puppy":    np.array([0.9, 0.9, 0.9, 0.0, 0.2, 0.3, 0.0]),
    "puppies":  np.array([0.9, 0.9, 0.9, 0.0, 0.2, 0.3, 0.0]),
    "kittens":  np.array([0.9, 0.8, 0.9, 0.0, 0.3, 0.0, 0.0]),
    "cats":     np.array([0.9, 0.8, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "dogs":     np.array([0.9, 0.9, 0.0, 0.0, 0.1, 0.3, 0.0]),
    "pets":     np.array([0.7, 1.0, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "animals":  np.array([1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "baby":     np.array([0.3, 0.2, 1.0, 0.0, 0.4, 0.0, 0.0]),
    "young":    np.array([0.2, 0.1, 0.9, 0.0, 0.1, 0.0, 0.0]),
    "adorable": np.array([0.4, 0.5, 0.6, 0.0, 0.5, 0.0, 0.0]),
    "loyal":    np.array([0.3, 0.6, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "friendly": np.array([0.3, 0.5, 0.0, 0.0, 0.2, 0.0, 0.0]),
    "sat":      np.array([0.0, 0.0, 0.0, 0.1, 0.1, 0.8, 0.0]),
    "chased":   np.array([0.0, 0.0, 0.0, 0.1, 0.0, 0.9, 0.0]),
    "mat":      np.array([0.0, 0.0, 0.0, 0.6, 0.8, 0.0, 0.0]),
    "garden":   np.array([0.0, 0.0, 0.0, 0.9, 0.2, 0.0, 0.7]),
    "flowers":  np.array([0.0, 0.0, 0.0, 0.5, 0.1, 0.0, 1.0]),
    "birds":    np.array([0.8, 0.1, 0.0, 0.4, 0.0, 0.3, 0.2]),
    "soft":     np.array([0.0, 0.0, 0.0, 0.1, 0.9, 0.0, 0.0]),
    "warm":     np.array([0.0, 0.0, 0.0, 0.1, 0.9, 0.0, 0.0]),
    "popular":  np.array([0.1, 0.3, 0.0, 0.0, 0.1, 0.0, 0.0]),
    "worldwide":np.array([0.0, 0.1, 0.0, 0.2, 0.0, 0.0, 0.0]),
    "most":     np.array([0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0]),
    "full":     np.array([0.0, 0.0, 0.0, 0.3, 0.0, 0.0, 0.2]),
}

# Show that similar words have similar vectors
print("Cosine similarity between word pairs:")
pairs = [("cat", "kitten"), ("cat", "dog"), ("cat", "garden"), ("puppy", "kitten"), ("mat", "soft")]
for w1, w2 in pairs:
    sim = 1 - cosine(word_vectors[w1], word_vectors[w2])
    print(f"  {w1:8s} <-> {w2:8s} = {sim:.3f}")

Cosine similarity between word pairs:
  cat      <-> kitten   = 0.807
  cat      <-> dog      = 0.968
  cat      <-> garden   = 0.028
  puppy    <-> kitten   = 0.978
  mat      <-> soft     = 0.861


In [11]:
def embed_text(text: str, vectors: dict) -> np.ndarray:
    """Create a document embedding by averaging the word vectors."""
    tokens = tokenize(text)
    vecs = [vectors[t] for t in tokens if t in vectors]
    if not vecs:
        return np.zeros(7)  # dimension of our vectors
    return np.mean(vecs, axis=0)

def embedding_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Search using average word-vector cosine similarity."""
    q_emb = embed_text(query, word_vectors)
    scores = []
    
    for idx, doc in enumerate(documents):
        d_emb = embed_text(doc, word_vectors)
        if np.linalg.norm(q_emb) == 0 or np.linalg.norm(d_emb) == 0:
            sim = 0.0
        else:
            sim = 1 - cosine(q_emb, d_emb)
        scores.append((idx, sim, doc))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# Now we can search with SEMANTICALLY related words!
print("Query: 'puppy' (no document contains the word 'puppy' directly)")
for idx, score, doc in embedding_search("puppy", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'young animal'")
for idx, score, doc in embedding_search("young animal", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'puppy' (no document contains the word 'puppy' directly)
  [5] score=0.981 | Puppies and kittens are adorable baby animals
  [2] score=0.978 | A kitten is a young cat
  [7] score=0.811 | Cats and dogs are the most popular pets worldwide

Query: 'young animal'
  [5] score=0.767 | Puppies and kittens are adorable baby animals
  [2] score=0.745 | A kitten is a young cat
  [7] score=0.229 | Cats and dogs are the most popular pets worldwide


**Key takeaway:** Word vectors let us find documents based on *meaning*, not just exact word overlap. Searching for "puppy" can match documents about "dogs" and "kittens" because those vectors are close in semantic space. However, averaging word vectors loses word order and can blur meaning for longer texts.

---

## 5. Neural Ranking (2018+)

With BERT (2018) and similar transformer models, we moved from comparing static word vectors to using deep neural networks that understand *context*. The same word can have different meanings in different sentences, and neural rankers capture this.

Real neural rankers use hundreds of millions of parameters. Here we'll build a **toy two-layer neural network** that learns to score query-document pairs, just to show the idea of a learned ranking function.

In [12]:
class ToyNeuralRanker:
    """A tiny neural network that scores query-document pairs.
    
    Architecture: concatenate query + doc embeddings -> hidden layer -> score
    This is a simplified cross-encoder pattern.
    """
    
    def __init__(self, input_dim: int = 7, hidden_dim: int = 8):
        np.random.seed(42)
        # Two embeddings concatenated = 2 * input_dim
        self.W1 = np.random.randn(2 * input_dim, hidden_dim) * 0.5
        self.b1 = np.zeros(hidden_dim)
        self.W2 = np.random.randn(hidden_dim, 1) * 0.5
        self.b2 = np.zeros(1)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, query_emb: np.ndarray, doc_emb: np.ndarray) -> float:
        """Score a query-document pair."""
        # Concatenate query and document embeddings
        x = np.concatenate([query_emb, doc_emb])
        # Hidden layer with ReLU
        h = self.relu(x @ self.W1 + self.b1)
        # Output score with sigmoid (0 to 1)
        score = self.sigmoid(h @ self.W2 + self.b2)
        return float(score[0])
    
    def train(self, examples: list[tuple[str, str, float]], epochs: int = 200, lr: float = 0.1):
        """Train on (query, document, relevance_label) triples."""
        print(f"Training on {len(examples)} examples for {epochs} epochs...")
        
        for epoch in range(epochs):
            total_loss = 0
            for query, doc, label in examples:
                q_emb = embed_text(query, word_vectors)
                d_emb = embed_text(doc, word_vectors)
                
                # Forward pass
                x = np.concatenate([q_emb, d_emb])
                h = self.relu(x @ self.W1 + self.b1)
                score = self.sigmoid(h @ self.W2 + self.b2)
                
                # Binary cross-entropy loss
                pred = float(score[0])
                loss = -(label * np.log(pred + 1e-8) + (1 - label) * np.log(1 - pred + 1e-8))
                total_loss += loss
                
                # Backprop (simplified gradient descent)
                d_score = pred - label  # gradient of BCE w.r.t. pre-sigmoid
                d_W2 = h.reshape(-1, 1) * d_score
                d_b2 = np.array([d_score])
                d_h = (self.W2.flatten() * d_score) * (h > 0).astype(float)
                d_W1 = x.reshape(-1, 1) @ d_h.reshape(1, -1)
                d_b1 = d_h
                
                # Update weights
                self.W2 -= lr * d_W2
                self.b2 -= lr * d_b2
                self.W1 -= lr * d_W1
                self.b1 -= lr * d_h
            
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}: loss = {total_loss / len(examples):.4f}")
    
    def search(self, query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
        """Rank documents for a query."""
        q_emb = embed_text(query, word_vectors)
        scores = []
        for idx, doc in enumerate(documents):
            d_emb = embed_text(doc, word_vectors)
            score = self.forward(q_emb, d_emb)
            scores.append((idx, score, doc))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

In [13]:
# Training data: (query, document, relevance)
# 1.0 = relevant, 0.0 = not relevant
training_data = [
    ("cat", "The cat sat on the mat", 1.0),
    ("cat", "The garden was full of flowers and birds", 0.0),
    ("dog", "Dogs are loyal and friendly pets", 1.0),
    ("dog", "The mat was soft and warm", 0.0),
    ("young animals", "Puppies and kittens are adorable baby animals", 1.0),
    ("young animals", "The garden was full of flowers and birds", 0.0),
    ("pets", "Cats and dogs are the most popular pets worldwide", 1.0),
    ("pets", "The mat was soft and warm", 0.0),
    ("kitten", "A kitten is a young cat", 1.0),
    ("kitten", "The dog chased the cat in the garden", 0.0),
]

ranker = ToyNeuralRanker()
ranker.train(training_data, epochs=200, lr=0.1)

Training on 10 examples for 200 epochs...
  Epoch 50: loss = 0.0317
  Epoch 100: loss = 0.0094
  Epoch 150: loss = 0.0051
  Epoch 200: loss = 0.0034


In [14]:
# Test the trained ranker
print("Query: 'cat'")
for idx, score, doc in ranker.search("cat", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'young animals'")
for idx, score, doc in ranker.search("young animals", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nQuery: 'puppy' (unseen in training!)")
for idx, score, doc in ranker.search("puppy", documents):
    print(f"  [{idx}] score={score:.3f} | {doc}")

Query: 'cat'
  [5] score=1.000 | Puppies and kittens are adorable baby animals
  [2] score=1.000 | A kitten is a young cat
  [3] score=1.000 | Dogs are loyal and friendly pets

Query: 'young animals'
  [5] score=1.000 | Puppies and kittens are adorable baby animals
  [2] score=1.000 | A kitten is a young cat
  [3] score=1.000 | Dogs are loyal and friendly pets

Query: 'puppy' (unseen in training!)
  [5] score=0.999 | Puppies and kittens are adorable baby animals
  [2] score=0.996 | A kitten is a young cat
  [3] score=0.922 | Dogs are loyal and friendly pets


**Key takeaway:** Neural rankers *learn* what makes a document relevant to a query from labeled examples. Even our tiny 2-layer network can generalize to unseen queries (like "puppy") because it operates on continuous embeddings rather than exact word matches. Real models like BERT go much further by understanding word order, context, and nuance.

---

## 6. Hybrid Retrieval (2020s)

The state of the art in retrieval today combines keyword-based and semantic methods. The insight is simple: **keyword search catches exact matches that vector search misses, and vector search catches semantic matches that keyword search misses.**

The most common fusion strategy is **Reciprocal Rank Fusion (RRF):** each retriever ranks the documents, and a document's combined score is the sum of `1 / (k + rank)` across all retrievers. This elegantly combines rankings without needing scores to be on the same scale.

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

Where `k` is a constant (typically 60) that prevents top-ranked documents from dominating.

In [15]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[tuple[int, float, str]]],
    k: int = 60,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """Combine multiple ranked lists using Reciprocal Rank Fusion."""
    rrf_scores = {}  # doc_idx -> score
    doc_texts = {}   # doc_idx -> text
    
    for ranked_list in ranked_lists:
        for rank, (doc_idx, _, doc_text) in enumerate(ranked_list):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0) + 1 / (k + rank + 1)
            doc_texts[doc_idx] = doc_text
    
    # Sort by combined RRF score
    results = [(idx, score, doc_texts[idx]) for idx, score in rrf_scores.items()]
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

def hybrid_search(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Hybrid retrieval combining TF-IDF (keyword) and embedding (semantic) search via RRF."""
    # Get rankings from both retrievers (return all docs so RRF has full picture)
    keyword_results = tfidf.search(query, top_k=len(documents))
    semantic_results = embedding_search(query, documents, top_k=len(documents))
    
    # Fuse with RRF
    return reciprocal_rank_fusion([keyword_results, semantic_results], top_k=top_k)

In [16]:
# Compare all three methods side by side
query = "puppy"

print(f"Query: '{query}'")
print(f"(Note: the word 'puppy' does NOT appear in any document)\n")

print("TF-IDF (keyword) results:")
kw_results = tfidf.search(query, top_k=3)
for idx, score, doc in kw_results:
    print(f"  [{idx}] score={score:.3f} | {doc}")
if not any(s > 0 for _, s, _ in kw_results):
    print("  (all scores are 0 — keyword search can't find 'puppy')")

print("\nEmbedding (semantic) results:")
for idx, score, doc in embedding_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nHybrid (TF-IDF + Embedding via RRF):")
for idx, score, doc in hybrid_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")

Query: 'puppy'
(Note: the word 'puppy' does NOT appear in any document)

TF-IDF (keyword) results:
  [0] score=0.000 | The cat sat on the mat
  [1] score=0.000 | The dog chased the cat in the garden
  [2] score=0.000 | A kitten is a young cat
  (all scores are 0 — keyword search can't find 'puppy')

Embedding (semantic) results:
  [5] score=0.981 | Puppies and kittens are adorable baby animals
  [2] score=0.978 | A kitten is a young cat
  [7] score=0.811 | Cats and dogs are the most popular pets worldwide

Hybrid (TF-IDF + Embedding via RRF):
  [2] score=0.0320 | A kitten is a young cat
  [0] score=0.0315 | The cat sat on the mat
  [5] score=0.0315 | Puppies and kittens are adorable baby animals


In [17]:
# Another example where hybrid shines: a query mixing a specific term + semantics
query = "friendly kitten"

print(f"Query: '{query}'\n")

print("TF-IDF (keyword):")
for idx, score, doc in tfidf.search(query, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nEmbedding (semantic):")
for idx, score, doc in embedding_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.3f} | {doc}")

print("\nHybrid (RRF fusion):")
for idx, score, doc in hybrid_search(query, documents, top_k=3):
    print(f"  [{idx}] score={score:.4f} | {doc}")

Query: 'friendly kitten'

TF-IDF (keyword):
  [3] score=0.346 | Dogs are loyal and friendly pets
  [2] score=0.258 | A kitten is a young cat
  [0] score=0.000 | The cat sat on the mat

Embedding (semantic):
  [2] score=0.985 | A kitten is a young cat
  [5] score=0.979 | Puppies and kittens are adorable baby animals
  [7] score=0.882 | Cats and dogs are the most popular pets worldwide

Hybrid (RRF fusion):
  [2] score=0.0325 | A kitten is a young cat
  [3] score=0.0320 | Dogs are loyal and friendly pets
  [5] score=0.0313 | Puppies and kittens are adorable baby animals


**Key takeaway:** Hybrid retrieval gets the best of both worlds. Keyword search handles exact term matches (great for names, codes, specific terminology), while semantic search handles conceptual similarity (great for synonyms and paraphrases). RRF is an elegant way to combine them without needing to normalize scores.

---

## Exercises

### Exercise 1: Weighted Hybrid Search

The RRF approach weights each retriever equally. But what if you know that for your use case, semantic search is more important than keyword search (or vice versa)?

Implement a `weighted_hybrid_search` function that accepts a `keyword_weight` and `semantic_weight` parameter. Multiply each retriever's RRF contribution by its weight before summing.

**Hint:** Modify the `reciprocal_rank_fusion` function to accept a list of weights alongside the ranked lists.

In [18]:
def weighted_rrf(
    ranked_lists: list[list[tuple[int, float, str]]],
    weights: list[float],
    k: int = 60,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """RRF with per-retriever weights."""
    # YOUR CODE HERE
    pass

def weighted_hybrid_search(
    query: str,
    documents: list[str],
    keyword_weight: float = 1.0,
    semantic_weight: float = 1.0,
    top_k: int = 3
) -> list[tuple[int, float, str]]:
    """Hybrid search with adjustable weights."""
    # YOUR CODE HERE
    pass

# Test: compare equal weighting vs keyword-heavy vs semantic-heavy
# weighted_hybrid_search("friendly kitten", documents, keyword_weight=1.0, semantic_weight=1.0)
# weighted_hybrid_search("friendly kitten", documents, keyword_weight=2.0, semantic_weight=0.5)
# weighted_hybrid_search("friendly kitten", documents, keyword_weight=0.5, semantic_weight=2.0)

### Exercise 2: Implement Normalized Levenshtein Similarity

Raw Levenshtein distance is hard to compare across strings of different lengths. "cat" -> "car" (distance 1) seems more similar than "elephant" -> "elephants" (distance 1), but the raw distance is the same.

Implement a `normalized_levenshtein_similarity` function that returns a value between 0 and 1:

$$\text{similarity}(s_1, s_2) = 1 - \frac{\text{levenshtein}(s_1, s_2)}{\max(|s_1|, |s_2|)}$$

Then update `levenshtein_search` to use this normalized similarity instead of raw distance.

In [19]:
def normalized_levenshtein_similarity(s1: str, s2: str) -> float:
    """Return a similarity score between 0 and 1."""
    # YOUR CODE HERE
    pass

# Test cases:
# normalized_levenshtein_similarity("cat", "car")       # Should be high (~0.67)
# normalized_levenshtein_similarity("cat", "garden")    # Should be low
# normalized_levenshtein_similarity("kitten", "sitting") # Should be moderate

### Exercise 3: Build a Three-Way Hybrid Retriever

We combined TF-IDF and embeddings above. Now add Jaccard similarity as a third retriever and fuse all three with RRF.

Then test it on the query `"adorable baby cat"` and compare the three-way hybrid against:
- TF-IDF alone
- Embedding search alone
- Two-way hybrid (TF-IDF + Embedding)

Does adding the third retriever change the ranking?

In [20]:
def three_way_hybrid(query: str, documents: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """Combine TF-IDF + Embedding + Jaccard via RRF."""
    # YOUR CODE HERE
    # 1. Get full rankings from tfidf.search, embedding_search, and jaccard_search
    # 2. Fuse all three with reciprocal_rank_fusion
    pass

# Test:
# query = "adorable baby cat"
# print("TF-IDF only:", tfidf.search(query))
# print("Embedding only:", embedding_search(query, documents))
# print("Two-way hybrid:", hybrid_search(query, documents))
# print("Three-way hybrid:", three_way_hybrid(query, documents))

---

## Summary

We've walked through six eras of information retrieval, building each technique from scratch:

| Method | Strength | Weakness |
|--------|----------|----------|
| **Levenshtein Distance** | Handles typos and spelling variants | Character-level only, slow on large corpora |
| **Jaccard Similarity** | Simple set overlap, fast to compute | Treats all words equally, ignores frequency |
| **TF-IDF** | Weighs important words higher, robust | No semantic understanding ("cat" ≠ "kitten") |
| **Word Vectors** | Captures semantic similarity | Loses word order, needs pre-trained vectors |
| **Neural Ranking** | Learns relevance from data, handles context | Needs labeled training data, expensive to run |
| **Hybrid Retrieval** | Best of keyword + semantic search | More complex to build and tune |

The key insight from this evolution: **no single method is best for everything.** Modern RAG systems use hybrid retrieval because real-world queries need both exact matching (product codes, names, specific terms) and semantic understanding (synonyms, paraphrases, conceptual similarity). The next notebooks in this section dive deeper into each technique — starting with BM25, then moving to reciprocal rank fusion and full hybrid pipelines.